<a href="https://colab.research.google.com/github/MichaelZiyu/207-course-notes/blob/main/canvascomposer_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎨 CanvasComposer-Lite
### A hands-on approximation of multi-subject controllable image generation

This notebook walks through the three core ideas behind **CanvasComposer** (SIGGRAPH Asia 2026):

1. Give each subject its own layer on a shared canvas
2. Use spatial position to build a structured text prompt
3. Generate a harmonised scene from that prompt

| Step | What happens |
|---|---|
| **0 — Setup** | Install packages, check GPU |
| **1 — Subjects** | Upload your photos (or use placeholders) |
| **2 — Canvas** | Drag subjects into a scene layout |
| **3 — Prompt** | Describe each subject; auto-build the prompt |
| **4 — Generate** | Run SDXL-Turbo locally on the T4 GPU |

> 💡 **Before you start:** go to **Runtime → Change runtime type → T4 GPU**  
> then run cells top-to-bottom with `Shift+Enter`.

---
## 0 · Setup

Run both cells below once. The install takes about 1 minute and is cached for the session.

In [1]:
# Check that the T4 GPU is available.
# If you see 'not found', go to Runtime → Change runtime type → T4 GPU.
import subprocess
r = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True
)
print("\u2705 GPU:", r.stdout.strip() if r.returncode == 0
      else "not found — go to Runtime → Change runtime type → T4 GPU")


✅ GPU: Tesla T4, 15360 MiB


In [ ]:
# Install the packages we need.
# diffusers   — loads and runs SDXL-Turbo for image generation
# transformers / accelerate / safetensors — required by diffusers
# ipywidgets  — powers the interactive prompt-builder UI
!pip install diffusers transformers accelerate safetensors ipywidgets -q
print("\u2705 Packages installed")


In [ ]:
# Standard library imports used throughout the notebook.
import os, io, base64, json, textwrap, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
from PIL import Image
import numpy as np
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# All generated files are saved here so you can download them later.
WORK_DIR = Path("canvas_output")
WORK_DIR.mkdir(exist_ok=True)
print("\u2705 Ready. Output folder:", WORK_DIR.resolve())


---
## 1 · Load Subjects

Each person in the final image is a **subject**. In CanvasComposer, each subject lives
on its own transparent (RGBA) layer — so the model never confuses whose features belong to whom.

Here we skip the background-removal step (it conflicts with Colab’s current numpy version)
and load subjects as rectangular crops. The canvas layout and generation still work identically.

**Pick one option:**
- **Option A (cell below)** — upload your own photos. Run the cell once per photo, or select multiple at once.
- **Option B (cell 1b)** — skip uploading and use coloured silhouette placeholders.

In [ ]:
from google.colab import files as colab_files

# Allow large phone photos without Pillow raising a safety warning.
Image.MAX_IMAGE_PIXELS = None

# Add HEIC support for iPhone photos.
!pip install pillow-heif -q
from pillow_heif import register_heif_opener
register_heif_opener()

# cutouts is the shared dict that every later cell reads from.
# Keys:   'subject_1', 'subject_2', ...
# Values: PIL RGBA images
cutouts = {}

def upload_one():
    """
    Opens a file picker for ONE photo and appends it to cutouts.
    Call this cell multiple times to add more subjects.
    """
    print(f"Pick a photo (subject_{len(cutouts)+1}):")
    uploaded = colab_files.upload()  # opens the file picker
    for fname, data in uploaded.items():
        name = f"subject_{len(cutouts)+1}"
        try:
            img = Image.open(io.BytesIO(data)).convert('RGB')
            # Resize very large photos to speed up later steps
            if max(img.size) > 1000:
                img.thumbnail((1000, 1000), Image.LANCZOS)
            cutouts[name] = img.convert('RGBA')
            cutouts[name].save(WORK_DIR / f'{name}.png')
            # Show a thumbnail so you can confirm the upload
            buf = io.BytesIO(); img.thumbnail((200, 200)); img.save(buf, 'PNG')
            b64 = base64.b64encode(buf.getvalue()).decode()
            display(HTML(f"<b>{name}</b> ({fname})<br>"
                         f"<img src='data:image/png;base64,{b64}'/>"))
        except Exception as e:
            print(f"  \u26a0\ufe0f  Could not open {fname}: {e}")
    print(f"\n\u2705 {len(cutouts)} subject(s) loaded so far: {list(cutouts.keys())}")

# Run this line. To add more subjects, run the cell again.
upload_one()


### 1b · No photos? Use placeholders

**Skip the cell above and run this one instead.**  
Creates four coloured silhouettes so you can explore the canvas and generation steps immediately.

In [ ]:
# Draw a simple person-shaped silhouette as an RGBA image.
def make_placeholder(color, size=(120, 200)):
    img = Image.new("RGBA", size, (0, 0, 0, 0))
    px = img.load(); w, h = size; r, g, b = color
    def ell(cx, cy, rx, ry):  # filled ellipse
        for y in range(max(0,cy-ry), min(h,cy+ry+1)):
            for x in range(max(0,cx-rx), min(w,cx+rx+1)):
                if ((x-cx)/rx)**2 + ((y-cy)/ry)**2 <= 1:
                    px[x, y] = (r, g, b, 255)
    def rect(x0, y0, x1, y1):  # filled rectangle
        for y in range(max(0,y0), min(h,y1)):
            for x in range(max(0,x0), min(w,x1)):
                px[x, y] = (r, g, b, 255)
    ell(w//2, int(h*.12), int(w*.16), int(h*.11))   # head
    rect(int(w*.28), int(h*.24), int(w*.72), int(h*.62)) # torso
    rect(int(w*.10), int(h*.24), int(w*.30), int(h*.52)) # left arm
    rect(int(w*.70), int(h*.24), int(w*.90), int(h*.52)) # right arm
    rect(int(w*.28), int(h*.60), int(w*.48), h)         # left leg
    rect(int(w*.52), int(h*.60), int(w*.72), h)         # right leg
    return img

cutouts = {}
for i, col in enumerate([(114,89,193), (6,182,212), (16,185,129), (245,158,11)]):
    name = f"subject_{i+1}"
    cutouts[name] = make_placeholder(col)
    cutouts[name].save(WORK_DIR / f'{name}.png')
    buf = io.BytesIO(); cutouts[name].thumbnail((150,150)); cutouts[name].save(buf,'PNG')
    b64 = base64.b64encode(buf.getvalue()).decode()
    display(HTML(f"<b>{name}</b><br><img src='data:image/png;base64,{b64}'/>"))
print(f"\n\u2705 {len(cutouts)} placeholder subjects ready")


---
## 2 · Canvas Layout 🖼️

This is the core CanvasComposer idea: instead of describing the layout in words,
you **drag subjects into position** and the notebook reads the coordinates.

**How to use:**
- **Drag** any subject card to position it in the scene
- **Purple ▪ handle** (bottom-right corner of each card) to resize — bigger = closer to camera
- **Background dropdown** to set the scene type
- Click **Snapshot layout** when you’re happy → a `canvas_layout` dict appears below
- Copy that dict into **cell 2b** (replacing the default values)

In [ ]:
# Shrink each subject image to a thumbnail for embedding in the canvas HTML.
def img_to_b64(img, max_px=120):
    t = img.copy(); t.thumbnail((max_px, max_px), Image.LANCZOS)
    buf = io.BytesIO(); t.convert("RGBA").save(buf, "PNG")
    return base64.b64encode(buf.getvalue()).decode()

# The canvas UI is a self-contained HTML+JS widget.
# __W__, __H__, __SJ__ are placeholders replaced at render time.
_TMPL = "<style>#ccw{font-family:system-ui;background:#0f0c1e;border-radius:10px;padding:14px;color:#f0eeff;display:inline-block}#ccc{position:relative;width:__W__px;height:__H__px;background:#1a1535;border:1px solid rgba(124,58,237,0.4);border-radius:8px;overflow:hidden;background-image:linear-gradient(rgba(124,58,237,0.05) 1px,transparent 1px),linear-gradient(90deg,rgba(124,58,237,0.05) 1px,transparent 1px);background-size:40px 40px}.ccs{position:absolute;cursor:grab;user-select:none;display:flex;flex-direction:column;align-items:center}.ccs:active{cursor:grabbing}.ccs img{border:2px solid transparent;border-radius:4px;transition:border-color 0.15s}.ccs:hover img{border-color:#a78bfa;box-shadow:0 0 12px rgba(124,58,237,0.6)}.ccl{font-size:10px;color:#9d8fcc;margin-top:2px;font-family:monospace}.cch{width:10px;height:10px;background:#7c3aed;border-radius:2px;cursor:se-resize;position:absolute;bottom:16px;right:0}#ccctrl{margin-top:10px;display:flex;gap:8px;align-items:center;flex-wrap:wrap}.ccb{font-size:12px;padding:6px 14px;border-radius:6px;border:none;cursor:pointer;font-weight:600}.pri{background:#7c3aed;color:#fff}.sec{background:#231d45;color:#9d8fcc;border:1px solid rgba(124,58,237,0.3)}select.sec{padding:6px 10px;font-size:12px}#ccout{font-size:11px;color:#6ee7b7;margin-top:10px;white-space:pre-wrap;display:none;max-width:__W__px}</style><div id=\"ccw\"><div style=\"font-size:12px;color:#9d8fcc;margin-bottom:8px\"><b style=\"color:#a78bfa\">Drag</b> to move &nbsp;&#183;&nbsp;<b style=\"color:#a78bfa\">&#9642; handle</b> to resize &nbsp;&#183;&nbsp;bigger = closer to camera</div><div id=\"ccc\"></div><div id=\"ccctrl\"><select class=\"sec\" onchange=\"setBg(this.value)\"><option value=\"\">No background</option><option value=\"park\">Park</option><option value=\"beach\">Beach</option><option value=\"city\">City rooftop</option><option value=\"studio\">Studio</option></select><button class=\"ccb sec\" onclick=\"resetLayout()\">Reset</button><button class=\"ccb pri\" onclick=\"snap()\">Snapshot layout</button><span id=\"ccs\" style=\"font-size:12px;color:#06b6d4;margin-left:4px\"></span></div><pre id=\"ccout\"></pre></div><script>var SJ=__SJ__,W=__W__,H=__H__;var cv=document.getElementById(\"ccc\");var bgs={park:\"linear-gradient(#7ec8e3 40%,#4caf50 40%)\",beach:\"linear-gradient(#87ceeb 50%,#f4d03f 50%)\",city:\"linear-gradient(#b0b8c1 55%,#7f8c8d 55%)\",studio:\"#f0f0f0\"};var els={},drag=null,rsz=null,dox=0,doy=0,rw0=0,rmx=0;function setBg(k){if(bgs[k]){cv.style.backgroundImage=bgs[k];cv.style.backgroundSize=\"cover\"}else{cv.style.background=\"#1a1535\";cv.style.backgroundImage=\"linear-gradient(rgba(124,58,237,0.05) 1px,transparent 1px),linear-gradient(90deg,rgba(124,58,237,0.05) 1px,transparent 1px)\";cv.style.backgroundSize=\"40px 40px\"}}function initSubjects(){var x=30;Object.entries(SJ).forEach(function(kv){var name=kv[0],b64=kv[1];var wrap=document.createElement(\"div\");wrap.className=\"ccs\";wrap.dataset.name=name;wrap.style.left=x+\"px\";wrap.style.top=\"80px\";var img=document.createElement(\"img\");img.src=\"data:image/png;base64,\"+b64;img.style.width=\"90px\";img.draggable=false;var lbl=document.createElement(\"div\");lbl.className=\"ccl\";lbl.textContent=name;var h=document.createElement(\"div\");h.className=\"cch\";wrap.appendChild(img);wrap.appendChild(lbl);wrap.appendChild(h);cv.appendChild(wrap);els[name]=wrap;wrap.addEventListener(\"mousedown\",function(e){if(e.target===h)return;drag=wrap;var r=cv.getBoundingClientRect();dox=e.clientX-r.left-parseInt(wrap.style.left);doy=e.clientY-r.top-parseInt(wrap.style.top);e.preventDefault();});h.addEventListener(\"mousedown\",function(e){rsz={el:wrap,img:img};rw0=img.offsetWidth;rmx=e.clientX;e.stopPropagation();e.preventDefault();});x+=130;});}document.addEventListener(\"mousemove\",function(e){if(drag){var r=cv.getBoundingClientRect();drag.style.left=Math.max(0,Math.min(W-60,e.clientX-r.left-dox))+\"px\";drag.style.top=Math.max(0,Math.min(H-60,e.clientY-r.top-doy))+\"px\";}if(rsz){rsz.img.style.width=Math.max(40,Math.min(220,rw0+(e.clientX-rmx)))+\"px\";}});document.addEventListener(\"mouseup\",function(){drag=null;rsz=null;});function resetLayout(){cv.innerHTML=\"\";els={};initSubjects();document.getElementById(\"ccout\").style.display=\"none\";document.getElementById(\"ccs\").textContent=\"\";}function snap(){var layout={};Object.entries(els).forEach(function(kv){var name=kv[0],el=kv[1];layout[name]={rel_x:parseFloat((parseInt(el.style.left)/W).toFixed(3)),rel_y:parseFloat((parseInt(el.style.top)/H).toFixed(3)),rel_size:parseFloat((el.querySelector(\"img\").offsetWidth/W).toFixed(3))};});window._layout=layout;var out=document.getElementById(\"ccout\");out.textContent=\"canvas_layout = \"+JSON.stringify(layout,null,2);out.style.display=\"block\";document.getElementById(\"ccs\").textContent=\"Snapshotted \u2713\";}initSubjects();</script>"

def build_canvas(cutouts, W=680, H=380):
    # Encode all subject thumbnails as base64 and inject into the template.
    sj = json.dumps({k: img_to_b64(v) for k, v in cutouts.items()})
    return _TMPL.replace('__W__', str(W)).replace('__H__', str(H)).replace('__SJ__', sj)

print("\u2705 Canvas ready")


In [ ]:
if not cutouts:
    print("\u26a0\ufe0f  No subjects loaded — run Part 1 first")
else:
    display(HTML(build_canvas(cutouts)))
    print("\n\U0001f446 Drag subjects into position, then click 'Snapshot layout'.")
    print("Copy the canvas_layout = {...} block that appears into cell 2b below.")


### 2b · Paste your layout here

After clicking **Snapshot layout** above, paste the printed `canvas_layout = {...}` here,
replacing the default values. Or just edit the numbers by hand:
- `rel_x` — horizontal position (0 = left edge, 1 = right edge)
- `rel_y` — vertical position (0 = top, 1 = bottom)
- `rel_size` — size relative to canvas width (0.2 = medium, 0.3 = large/foreground)

In [ ]:
# Paste the snapshotted layout here, or edit these defaults.
canvas_layout = {
    "subject_1": {"rel_x": 0.08, "rel_y": 0.20, "rel_size": 0.20},
    "subject_2": {"rel_x": 0.35, "rel_y": 0.18, "rel_size": 0.23},
    "subject_3": {"rel_x": 0.62, "rel_y": 0.22, "rel_size": 0.20},
    "subject_4": {"rel_x": 0.78, "rel_y": 0.28, "rel_size": 0.16},
}

# Only keep entries that have a matching loaded subject.
canvas_layout = {k: v for k, v in canvas_layout.items() if k in cutouts}
print(f"Layout confirmed: {list(canvas_layout.keys())}")


---
## 3 · Prompt Builder 📝

CanvasComposer conditions on **visual tokens per subject**. Here we approximate that
by converting canvas positions into natural language: *left of frame, in the foreground*, etc.

**Fill in a description for each subject, set the scene and style, then click Build prompt.**

In [ ]:
subject_descriptions = {}   # filled in when you click 'Build prompt'
final_prompt = {"value": ""}  # the assembled prompt, read by Part 4

# Maps relative canvas position → spatial language for the prompt.
H_POS = [("left of frame",   (0, 0.33)),
         ("centre of frame", (0.33, 0.67)),
         ("right of frame",  (0.67, 1))]
DEPTH = [("in the foreground", (0.18, 1)),
         ("in the midground",  (0.10, 0.18)),
         ("in the background", (0, 0.10))]

def to_h(rx):   # horizontal bucket
    for label, (lo, hi) in H_POS:
        if lo <= rx < hi: return label
    return "centre of frame"

def to_d(rs):   # depth bucket based on relative size
    for label, (lo, hi) in DEPTH:
        if lo <= rs < hi: return label
    return "in the scene"

# Build one text field per loaded subject.
rows, fields = [], {}
for name in canvas_layout:
    lbl = widgets.Label(value=f"{name}:", layout=widgets.Layout(width="90px"))
    txt = widgets.Text(
        placeholder="e.g. young woman, red jacket, curly hair",
        layout=widgets.Layout(width="380px")
    )
    fields[name] = txt
    rows.append(widgets.HBox([lbl, txt]))

scene_w = widgets.Text(
    value="friends at a rooftop party, golden hour, warm light",
    description="Scene:", layout=widgets.Layout(width="480px"),
    style={"description_width": "55px"}
)
style_w = widgets.Text(
    value="photorealistic, 35mm film, shallow depth of field",
    description="Style:", layout=widgets.Layout(width="480px"),
    style={"description_width": "55px"}
)
btn = widgets.Button(description="Build prompt", button_style="primary")
out_w = widgets.Output()

def on_build(_):
    with out_w:
        clear_output()
        subject_descriptions.clear()
        for name, w in fields.items():
            subject_descriptions[name] = w.value.strip() or f"person ({name})"

        # Sort subjects left-to-right so the prompt reads naturally.
        names = sorted(canvas_layout, key=lambda n: canvas_layout[n].get('rel_x', 0.5))
        parts = []
        for name in names:
            if name not in subject_descriptions: continue
            lay = canvas_layout[name]
            h   = to_h(lay.get('rel_x', 0.5))
            d   = to_d(lay.get('rel_size', 0.15))
            parts.append(f"[{name}: {subject_descriptions[name]}, {h}, {d}]")

        prompt = (
            f"Group photo of {len(parts)} people: {', '.join(parts)}. "
            f"Setting: {scene_w.value}. Style: {style_w.value}. "
            "All subjects fully visible, natural poses, harmonious lighting, "
            "photographic quality."
        )
        final_prompt["value"] = prompt
        print("─" * 70)
        print(textwrap.fill(prompt, 80))
        print("─" * 70)
        print(f"{len(prompt)} characters | {len(parts)} subjects")

btn.on_click(on_build)
display(widgets.VBox(rows + [scene_w, style_w, btn, out_w]))


---
## 4 · Generate 🚀

We use **SDXL-Turbo** (Stability AI) — a distilled model that generates in 4 steps (~15 sec on T4).
No login or API key required. Weights (~6 GB) download once and are cached for the session.

> ⏱️ **First run: ~3 minutes** to download. Every run after that: ~15 seconds.

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.float16 if device == "cuda" else torch.float32
print(f"Loading SDXL-Turbo on {device}...")
print("First run: ~3 min to download weights (cached after that).")

pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=dtype,
    variant="fp16" if device == "cuda" else None,
).to(device)
pipe.enable_attention_slicing()  # reduces GPU memory usage
print(f"\n\u2705 SDXL-Turbo ready on {device}")


In [ ]:
# Generate one image from the prompt built in Part 3.
# seed  — change this number to get a different result with the same prompt
# steps — SDXL-Turbo is distilled: 4 steps is optimal (more doesn't help)
def generate(prompt, seed=42, steps=4, w=512, h=512):
    g = torch.Generator(device=device).manual_seed(seed)
    return pipe(
        prompt=prompt,
        num_inference_steps=steps,
        guidance_scale=0.0,  # Turbo doesn't use classifier-free guidance
        width=w, height=h,
        generator=g
    ).images[0]

prompt = final_prompt["value"]
if not prompt:
    print("\u26a0\ufe0f  Build the prompt in Part 3 first, then re-run this cell.")
else:
    print("Generating... (~15 sec)")
    img = generate(prompt)
    img.save(WORK_DIR / "generated.png")
    buf = io.BytesIO(); t = img.copy(); t.thumbnail((700, 700)); t.save(buf, "PNG")
    b64 = base64.b64encode(buf.getvalue()).decode()
    display(HTML(
        f"<img src='data:image/png;base64,{b64}' "
        "style='max-width:700px;border-radius:8px'/>"
    ))
    print("\n\u2705 Saved to canvas_output/generated.png")
    print("To download: click the folder icon in the left sidebar.")


In [ ]:
# Generate the same prompt with 3 different random seeds.
# This shows how much variation exists within a single prompt —
# a good discussion point for the workshop.
prompt = final_prompt["value"]
if not prompt:
    print("\u26a0\ufe0f  Build the prompt in Part 3 first.")
else:
    html_out = ""
    for seed in [0, 7, 42]:
        print(f"  Generating seed={seed}...")
        img = generate(prompt, seed=seed)
        img.save(WORK_DIR / f"generated_seed{seed}.png")
        buf = io.BytesIO(); t = img.copy(); t.thumbnail((300, 300)); t.save(buf, "PNG")
        b64 = base64.b64encode(buf.getvalue()).decode()
        html_out += (
            f"<div style='display:inline-block;margin:6px;text-align:center'>"
            f"<img src='data:image/png;base64,{b64}' style='border-radius:6px'/>"
            f"<div style='font-size:11px;color:#666'>seed={seed}</div></div>"
        )
    display(HTML(html_out))
    print("\n\u2705 All three saved to canvas_output/")


---
## 🔍 What the real paper does differently

| Step | This notebook | CanvasComposer (paper) |
|---|---|---|
| Subject isolation | Rectangular crop (no bg removal) | RGBA layer per subject |
| Subject conditioning | Spatial words in text prompt | Visual tokens injected into DiT attention |
| Identity preservation | Relies on text description | Cross-reference training across multiple photos |
| Occlusion | Handled in text only | Separate layers prevent cross-contamination |
| Token efficiency | N/A | Transparent-pixel pruning: 16,384 → 2,200 tokens |

**What this notebook does capture:**
- The one-layer-per-subject mental model
- Spatial layout → structured language (left/centre/right, foreground/background)
- The iterative arrange-then-generate workflow

**Paper links:**
- CanvasComposer: https://snap-research.github.io/canvascomposer/
- ProxyPose: https://ruihangzhang97.github.io/proxypose/